# Module 14: Training NER Models


## 🏷️ Training a Custom Named Entity Recognizer

Let's say we want to build a model that automatically recognizes Gadgets (`GADGET`). 
The standard spaCy model doesn't know what a `GADGET` is. We have to train it!

First, we need raw data. In Python, this is usually stored as a list of tuples: `(text, annotations_dictionary)`.


In [ ]:
import spacy
from spacy.tokens import DocBin

# 1. Define our raw training data
# Format: (text, {"entities": [(start_char, end_char, LABEL)]})
TRAIN_DATA = [
    ("I bought a new iPhone 14 yesterday.", {"entities": [(15, 24, "GADGET")]}),
    ("The Samsung Galaxy is quite expensive.", {"entities": [(4, 18, "GADGET")]}),
    ("My Macbook Pro crashed.", {"entities": [(3, 14, "GADGET")]}),
    ("I need to charge my iPad.", {"entities": [(20, 24, "GADGET")]}),
    ("He dropped his Apple Watch on the floor.", {"entities": [(15, 26, "GADGET")]})
]

# We also need evaluation data (dev data) so the model can test itself
DEV_DATA = [
    ("The new iPhone is amazing.", {"entities": [(8, 14, "GADGET")]}),
    ("Can you fix this Samsung Galaxy?", {"entities": [(17, 31, "GADGET")]})
]


<br><br>

---

<br><br>


## 💾 Converting to `.spacy` Format

Now we must convert these raw dictionaries into `DocBin` objects and save them to disk as `train.spacy` and `dev.spacy`.


In [ ]:
def convert_to_spacy_format(data, filename):
    nlp = spacy.blank("en") # We use a blank model to tokenize the text
    db = DocBin()
    
    for text, annotations in data:
        doc = nlp.make_doc(text)
        ents = []
        for start, end, label in annotations["entities"]:
            # Create a character span for each entity
            span = doc.char_span(start, end, label=label)
            if span is None:
                print(f"Skipping entity in text: '{text}' because the character indices don't align with token boundaries.")
            else:
                ents.append(span)
                
        doc.ents = ents
        db.add(doc)
        
    db.to_disk(filename)
    print(f"Saved {len(data)} examples to {filename}")

# Run the conversion
convert_to_spacy_format(TRAIN_DATA, "train.spacy")
convert_to_spacy_format(DEV_DATA, "dev.spacy")


<br><br>

---

<br><br>


## 🔄 The Training Commands

Now that the data is prepared, you would run the following commands in your terminal to actually train the model. 

*Note: We are not running these directly in the notebook because training is best done via the CLI, but this is exactly what you would type!*


In [ ]:
"""
Step 1: Generate the base config for an English NER model
!python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency

Step 2: Train the model!
!python -m spacy train config.cfg --output ./models --paths.train ./train.spacy --paths.dev ./dev.spacy
"""


<br><br>

---

<br><br>


## ⚠️ Adding Labels to Existing Models (Catastrophic Forgetting)

What if you don't want to start from a `spacy.blank("en")` model? What if you want to use `en_core_web_sm` (which already knows about `PERSON` and `ORG`) and just *add* `GADGET` to it?

If you take the `en_core_web_sm` model and train it using ONLY our 5 `GADGET` sentences, the neural network will overwrite its old weights to optimize for `GADGET`. It will completely forget what a `PERSON` or `ORG` is!

This is called **Catastrophic Forgetting**.

### The Solution:
If you want to add a label to an existing model, your training data **MUST** include examples of the old labels as well! You must remind the model what a `PERSON` is while teaching it what a `GADGET` is. This is why training entirely new categories from a blank model is often easier and safer.
